# BO benchmark on the source-model black box
Kumo large and Causilo (regressors and classifier arms) against a BoTorch GP, two acquisition functions, with and without context from the real machines. Everything is saved to Drive after each evaluation: when the time budget is reached or Colab disconnects, run the cells again and it continues.

In [ ]:
# 1. Setup: Drive, repo, packages. Safe to re-run after a disconnect.
REPO_URL = "https://github.com/josefullrich/ASML.git"   # private: add a GITHUB_TOKEN secret in Colab
BRANCH   = "main"
DRIVE    = "/content/drive/MyDrive/ASML"                 # holds data/ and results/ (not in git)

import os, subprocess
from google.colab import drive, userdata
drive.mount("/content/drive")
def secret(name):
    try: return userdata.get(name)
    except Exception: return None
tok = secret("GITHUB_TOKEN")
url = REPO_URL.replace("https://", f"https://{tok}@") if tok else REPO_URL
if not os.path.isdir("/content/ASML"):
    subprocess.run(["git", "clone", "-b", BRANCH, url, "/content/ASML"], check=True)
else:
    subprocess.run(["git", "-C", "/content/ASML", "pull"], check=False)
if not os.path.isdir("/content/tfm_onnx_in_matlab"):   # weighted classifier graphs
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/iwoodsawyer/tfm_onnx_in_matlab.git", "/content/tfm_onnx_in_matlab"], check=True)
!pip -q install botorch bocode causilo structured-data-models plotly onnx onnxscript statsmodels
if secret("HF_TOKEN"):
    os.environ["HF_TOKEN"] = secret("HF_TOKEN")
os.makedirs(f"{DRIVE}/results/bo_bench", exist_ok=True)
%cd /content/ASML/notebooks
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (the TFMs will be very slow)")

In [ ]:
# 2. Settings. Anything in bo_bench/config.py can go here.
import json
SETTINGS = dict(
    results_dir = f"{DRIVE}/results/bo_bench",
    data_dir    = f"{DRIVE}/data",
    srcmdl_dir  = "/content/ASML/src",          # folder containing srcmdl.py
    ivo_repo    = "/content/tfm_onnx_in_matlab",
    budget_hours = 11.0,                        # stops cleanly and saves; run the cell again to continue
    models = ["kumo", "causilo"],
    kumo_size = "large",
    n_estimators = 1,
    seeds = [1, 2],                             # black-box seeds, shared by every run
    n_iter = 350,
    n_candidates = 4096,
    contexts = ["none", "machines"],
    context_per_machine = 2,
    context_input_scaling = "minmax",           # minmax | zscore | log | none
    cost_option = 2,                            # the MATLAB example's logit penalty
    extra_scatter_rho = 0.0,                    # >0 adds run-to-run yield scatter to the simulator
)
json.dump(SETTINGS, open("/content/bo_bench_settings.json", "w"), indent=1)
print(json.dumps(SETTINGS, indent=1))

**3. Time check.** Runs a few BO steps of each arm and projects the total, so you know how many sessions it needs before starting.

In [ ]:
!python -m bo_bench.run estimate --config /content/bo_bench_settings.json

**4. Run.** Prediction benchmark first, then the BO runs, then the report. Re-run this cell to resume.

In [ ]:
!python -m bo_bench.run all --config /content/bo_bench_settings.json

**5. Report.** Can be built at any time, also from partial results.

In [ ]:
!python -m bo_bench.run report --config /content/bo_bench_settings.json
from google.colab import files
files.download(f"{DRIVE}/results/bo_bench/srcmdl/report.html")

**Optional: carry the saved state as a file** instead of using Drive. `pack` writes a zip; on a fresh machine upload it and `unpack` before running.

In [ ]:
# !python -m bo_bench.run pack   --config /content/bo_bench_settings.json --zip /content/bo_bench_state.zip
# !python -m bo_bench.run unpack --config /content/bo_bench_settings.json --zip /content/bo_bench_state.zip